In [ ]:
# we're just using phetk for the mapping file, the phecode module otherwise makes mapping complicated
# (needs notion of patients, code time, and may exclude codes for in the phewas tooling)

# !pip install phetk simple-icd-10-cm

import os
import ast

import phetk
import numpy as np
import pandas as pd
import simple_icd_10_cm as cm
from pqdm.processes import pqdm
from tqdm import tqdm
from wfdb import rdsamp

### Get Phecode Mappings

These are the phecodes used in:

**Prototype Learning to Create Refined Interpretable Digital Phenotypes from ECGs** (Sethi et al. 2026)

https://doi.org/10.1142/9789819824755_0049

In [ ]:
# from the PSB paper, slight corrections:
# - pad leading 0s for 38 and 38.1
# - trim trailing 0s for 38.0, 260.0, 501.0, 507.0, 572.0
# - remove cardiac arrest 427.42 (duplicate with MDS-ED task)
target_phecodes = [
    "038", "038.1", "260", "260.2", "275.53", "276.11", "276.6", "284.1", "286.7", "290.2",
    "317.11", "395.2", "395.6", "411.2", "411.8", "415.2", "425.1", "427.12", "427.21", "427.22",
    # "427.42",
    "428.1", "428.3", "428.4", "501", "507", "509.1", "572", "585.32", "994.2",
]

In [ ]:
# get the global mappings (looks like PSB paper uses phecode verison 1.2)
df = pd.read_csv(
    os.path.join(os.path.dirname(phetk.__file__), "phecode", "phecode12.csv"),
    dtype={
        "phecode": str,
        "ICD": str,
        "flag": int,
        "exclude_range": str,
        "phecode_unrolled": str,
    },
)

# we only need ICD10 since we're using these with MDS-ED which has converted to ICD10
df = df[df["flag"] == 10]

# filter to just the targets
df = df[df["phecode"].isin(target_phecodes)]

In [ ]:
phe_to_name = (
    df[["phecode", "phecode_string"]]
    .drop_duplicates()
    .set_index("phecode")["phecode_string"]
    .str.capitalize()
    .loc[target_phecodes]
    .to_dict()
)

phe_to_icds = (
    df.groupby("phecode")["ICD"]
    .apply(lambda x: list(set(x)))
    .loc[target_phecodes]
    .to_dict()
)

### Cleanup MDS-ED Data

MDS-ED provides some nice predictive tasks for ECGs collected in the ED.

These ECGs are collected within the first 90 minutes of the ED encounter.

See: https://physionet.org/content/multimodal-emergency-benchmark/1.0.0/

In [ ]:
mds_ed_path = "/opt/gpudata/mimic/mimic-iv-ext-mds-ed-v1.0.0"
general_cols = [
    "general_file_name",
    "general_study_id",
    "general_subject_id",
    "general_ed_diag_ed",
    "general_ed_diag_hosp",
    "general_ecg_no_within_stay",
    "general_strat_fold",
    "general_ecg_time",
    "general_intime",
    "general_outtime",
    "general_90min",
    "demographics_gender",
    "demographics_age",
]
prediction_cols =[
    "deterioration_severe_hypoxemia",
    "deterioration_ecmo",
    "deterioration_vasopressors",
    "deterioration_inotropes",
    "deterioration_mechanical_ventilation",
    "deterioration_cardiac_arrest",
    "deterioration_icu_24h",
    "deterioration_icu_stay",
    "deterioration_mortality_1d",
    "deterioration_mortality_7d",
    "deterioration_mortality_28d",
    "deterioration_mortality_90d",
    "deterioration_mortality_180d",
    "deterioration_mortality_365d",
    "deterioration_mortality_stay",
]
mds_ed = pd.read_csv(os.path.join(mds_ed_path, "mds_ed.csv"), usecols=general_cols+prediction_cols)
mds_ed["general_file_name"] = mds_ed["general_file_name"].str.replace("mimic-iv-ecg-diagnostic-electrocardiogram-matched-subset-1.0/", "")

#### Exclude ECGs with NaNs

In [ ]:
dataset_path = "/opt/gpudata/ecg/mimic-iv-ecg"
X = []
for fpath in tqdm(mds_ed["general_file_name"]):
    x, _ = rdsamp(os.path.join(dataset_path, fpath))
    X.append(x)
X = np.stack(X) # (N, T, 12)

In [ ]:
# the stratified folds must not split a patient's samples
assert not mds_ed[["general_subject_id", "general_strat_fold"]].drop_duplicates()["general_subject_id"].duplicated().any()

# only use samples which have well defined endpoints
# NOTE: from the MDS-ED authors: "we also include a special token (-999) which refers to the 
# exclusion of specific target labels due to the clinical workflow absence of the entire patient visit."
# see: https://physionet.org/content/multimodal-emergency-benchmark/1.0.0/
outcome_mask = ~(mds_ed[prediction_cols] == -999).any(axis=1)
print(f"{(~outcome_mask).sum()} ECGs have missing labels")

# only use samples which were collected in the ED
# NOTE: this is stricter than presumably what the MDS-ED authors did which is require ECG time <= ED intime + 90min
# can check: (mds_ed["general_ecg_time"] <= mds_ed["general_90min"]).all()
# should only be a few samples anyways
timing_mask = mds_ed["general_ecg_time"] <= mds_ed["general_outtime"]
print(f"{(~timing_mask).sum()} ECGs were collected after leaving the ED")

# only use samples which have no nan values in the ECG
no_nan_mask = np.isnan(X).sum(axis=(1, 2)) == 0
print(f"{(~no_nan_mask).sum()} ECGs have nan values in the waveforms")

mask = outcome_mask & timing_mask & no_nan_mask
df = mds_ed.loc[mask].reset_index(drop=True)
print(f"Excluded {(~mask).sum()} ECGs based on the intersection of the above criteria")

### Prepare Labels

We use a mixture of endpoints:
* outcome diagnosis: cardiac, non-cardiac, and more specific phenotyping
* ICU admission: within 24 hours of ED encounter or at any point of the ED stay
* hospital mortality: 7 forecasting windows
* clinical deterioration: hypoxemia, ecmo, vasopressors, inotropes, mechanical ventilation, cardiac arrest

These endpoints are defined by:
* Benchmarking ECG FMs: a reality check across clinical tasks (Al-Masud et al. 2026)
* Prototype Learning to Create Refined Interpretable Digital Phenotypes from ECGs (Sethi et al. 2026)

In [ ]:
kwargs = [
    {
        "ed_diag": ed_diag,
        "hosp_diag": hosp_diag,
    }
    for ed_diag, hosp_diag in zip(
        df["general_ed_diag_ed"],
        df["general_ed_diag_hosp"],
    )
]

def compute_icd_labels(ed_diag, hosp_diag) -> np.ndarray:
    # list order: cardiac, non-cardiac, phecodes
    labels = np.zeros(2 + len(target_phecodes))
    dxs = set(ast.literal_eval(ed_diag)) | set(ast.literal_eval(hosp_diag))
    for dx in dxs:
        if not cm.is_valid_item(dx):
            continue  # don't count anything if invalid

        if cm.is_descendant(dx, "9"):
            # cardiac chapter 9
            labels[0] = 1
        else:
            labels[1] = 1

        for j, cands in enumerate(phe_to_icds.values()):
            for cand in cands:
                if cm.add_dot(dx) == cm.add_dot(cand) or cm.is_descendant(dx, cand):
                    labels[2 + j] = 1
    return labels


cardiac_noncardiac_phecodes = pqdm(
    kwargs, compute_icd_labels, n_jobs=40, argument_type="kwargs" # type: ignore
)

In [ ]:
cardiac_noncardiac_phecode_names = ["Cardiac diagnosis", "Non-cardiac diagnosis"] + list(phe_to_name.values())

In [ ]:
df[cardiac_noncardiac_phecode_names] = np.stack(cardiac_noncardiac_phecodes)

In [ ]:
rename = {
    "general_file_name": "file_name",
    "general_study_id": "study_id",
    "general_subject_id": "subject_id",
    "demographics_age": "age",
    "demographics_gender": "gender",
    "general_ecg_no_within_stay": "ecg_no_within_stay",
    "general_strat_fold": "strat_fold",
    "deterioration_severe_hypoxemia": "Severe hypoxemia",
    "deterioration_ecmo": "ECMO",
    "deterioration_vasopressors": "Vasopressors",
    "deterioration_inotropes": "Inotropes",
    "deterioration_mechanical_ventilation": "Mechanical ventilation",
    "deterioration_cardiac_arrest": "Cardiac arrest",
    "deterioration_icu_24h": "ICU admit within 24 hours",
    "deterioration_icu_stay": "ICU admit",
    # not quite sure what the start time is for these prediction horizons
    "deterioration_mortality_1d": "Mortality 24 hours",
    "deterioration_mortality_7d": "Mortality 7 days",
    "deterioration_mortality_28d": "Mortality 28 days",
    "deterioration_mortality_90d": "Mortality 90 days",
    "deterioration_mortality_180d": "Mortality 180 days",
    "deterioration_mortality_365d": "Mortality 365 days",
    "deterioration_mortality_stay": "Mortality stay",
}

df = df.rename(columns=rename)

In [ ]:
# do some final cleanup of the dataframe
final_cols = list(rename.values()) + cardiac_noncardiac_phecode_names
non_label_cols = ["file_name", "study_id", "subject_id", "ecg_no_within_stay", "strat_fold", "age", "gender"]
label_cols = [c for c in final_cols if c not in non_label_cols]
df = df[final_cols]
for label_col in label_cols:
    df[label_col] = df[label_col].astype(int)
assert df[label_cols].isin({0, 1}).all(axis=None)

In [ ]:
# fold 18/19 for val/test based on author recommendations
df["split"] = "train"
df.loc[df["strat_fold"] == 18, "split"] = "val"
df.loc[df["strat_fold"] == 19, "split"] = "test"

# do some cleanup to enforce single ECG per patient in val/test
df = df.loc[
    (df["split"] == "train")
    | ((df["split"].isin(["val", "test"])) & (df["ecg_no_within_stay"] == 0))
].reset_index(drop=True)

In [ ]:
base_path = "/opt/gpudata/ecg/mimic-iv-ecg"
df.to_csv(os.path.join(base_path, "ed-ecgs.csv.NEW"), index=False)

### For Defines

In [ ]:
label_cols

In [ ]:
# compute waveform normalizations over train data

dataset_path = "/opt/gpudata/ecg/mimic-iv-ecg"
df = pd.read_csv(os.path.join(dataset_path, "ed-ecgs.csv"))
train_df = df[df["split"] == "train"]

X = []
for fpath in tqdm(train_df["file_name"]):
    x, _ = rdsamp(os.path.join(dataset_path, fpath))
    X.append(x)
X = np.stack(X)

In [ ]:
lowers, uppers = np.percentile(X, [0.1, 99.9], axis=(0, 1))
display(lowers.tolist())
display(uppers.tolist())

In [ ]:
X_clipped = np.clip(X, lowers, uppers)
means = X_clipped.mean(axis=(0, 1))
stds = X_clipped.std(axis=(0, 1))
display(means.tolist())
display(stds.tolist())